In [1]:
# %%
from pathlib import Path
import json


PROJECT_ROOT = Path(
    r"C:\Users\abd93000\PycharmProjects\cnt_project_v2"
)

RUNS_ROOT = (
    PROJECT_ROOT
    / "global_outputs"
    / "runs"
)

RUN_NAME = (
    "cpsam_v2_legacy_seed42_best_validation_optimized"
)

RUN_DIR = (
    RUNS_ROOT
    / RUN_NAME
)

PREDICTION_RLE_JSON = (
    RUN_DIR
    / "inference"
    / "predicted_annotations_rle.json"
)


print("Run directory:")
print(RUN_DIR)

print("\nPrediction JSON:")
print(PREDICTION_RLE_JSON)

print(
    "\nPrediction JSON exists:",
    PREDICTION_RLE_JSON.is_file(),
)


if not PREDICTION_RLE_JSON.is_file():
    raise FileNotFoundError(
        f"Prediction RLE JSON not found:\n"
        f"{PREDICTION_RLE_JSON}"
    )


with open(
    PREDICTION_RLE_JSON,
    "r",
    encoding="utf-8",
) as file:
    prediction_coco = json.load(file)


print("\nCOCO keys:")
print(
    prediction_coco.keys()
)

print(
    "\nNumber of images:",
    len(
        prediction_coco.get(
            "images",
            [],
        )
    ),
)

print(
    "Number of annotations:",
    len(
        prediction_coco.get(
            "annotations",
            [],
        )
    ),
)

Run directory:
C:\Users\abd93000\PycharmProjects\cnt_project_v2\global_outputs\runs\cpsam_v2_legacy_seed42_best_validation_optimized

Prediction JSON:
C:\Users\abd93000\PycharmProjects\cnt_project_v2\global_outputs\runs\cpsam_v2_legacy_seed42_best_validation_optimized\inference\predicted_annotations_rle.json

Prediction JSON exists: True

COCO keys:
dict_keys(['images', 'annotations', 'categories'])

Number of images: 30
Number of annotations: 2853


In [2]:
# %%
TARGET_STEM = (
    "400-1093-w11-c1p1-befo3-10sp-15flow_left"
)


matching_images = []

for image_info in prediction_coco.get(
    "images",
    [],
):
    file_name = str(
        image_info.get(
            "file_name",
            "",
        )
    )

    file_stem = Path(
        file_name
    ).stem

    if file_stem == TARGET_STEM:
        matching_images.append(
            image_info
        )


print(
    f"Target image: {TARGET_STEM}"
)

print(
    f"Found: {bool(matching_images)}"
)

print(
    f"Number of matching image entries: "
    f"{len(matching_images)}"
)


for image_info in matching_images:
    print(
        "\nMatching COCO image entry:"
    )

    print(
        json.dumps(
            image_info,
            indent=2,
        )
    )

    image_id = image_info[
        "id"
    ]

    image_annotations = [
        annotation
        for annotation
        in prediction_coco.get(
            "annotations",
            [],
        )
        if annotation.get(
            "image_id"
        ) == image_id
    ]

    print(
        "\nImage ID:",
        image_id,
    )

    print(
        "Number of predicted objects:",
        len(
            image_annotations
        ),
    )

Target image: 400-1093-w11-c1p1-befo3-10sp-15flow_left
Found: True
Number of matching image entries: 1

Matching COCO image entry:
{
  "id": 2,
  "file_name": "400-1093-w11-c1p1-befo3-10sp-15flow_left.tif",
  "width": 256,
  "height": 256
}

Image ID: 2
Number of predicted objects: 168


In [3]:
# %%
import sys
from pathlib import Path


SRC_DIR = (
    PROJECT_ROOT
    / "src"
)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_DIR),
    )


from cnt_project.visualizations.overlays.coco import (
    visualize_json_annotations_as_polygons_svg,
)


# ---------------------------------------------------------
# Find polygon prediction JSON
# ---------------------------------------------------------

candidate_paths = [
    RUN_DIR
    / "inference"
    / "predicted_annotations_poly.json",

    RUN_DIR
    / "predicted_annotations_poly.json",
]


PREDICTION_POLY_JSON = None

for candidate_path in candidate_paths:
    if candidate_path.is_file():
        PREDICTION_POLY_JSON = candidate_path
        break


if PREDICTION_POLY_JSON is None:
    raise FileNotFoundError(
        "Could not find predicted_annotations_poly.json "
        f"under:\n{RUN_DIR}"
    )


IMAGE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cnt_segmentation"
    / "images"
)


print("Polygon prediction:")
print(PREDICTION_POLY_JSON)

print("\nImage directory:")
print(IMAGE_DIR)

print(
    "\nImage directory exists:",
    IMAGE_DIR.is_dir(),
)

Polygon prediction:
C:\Users\abd93000\PycharmProjects\cnt_project_v2\global_outputs\runs\cpsam_v2_legacy_seed42_best_validation_optimized\inference\predicted_annotations_poly.json

Image directory:
C:\Users\abd93000\PycharmProjects\cnt_project_v2\data\cnt_segmentation\images

Image directory exists: True


In [4]:
# %%
import json


TARGET_STEM = (
    "400-1093-w11-c1p1-befo3-10sp-15flow_left"
)


with open(
    PREDICTION_POLY_JSON,
    "r",
    encoding="utf-8",
) as file:
    prediction_poly_coco = json.load(
        file
    )


# ---------------------------------------------------------
# Find target image
# ---------------------------------------------------------

target_images = [
    image_info
    for image_info
    in prediction_poly_coco["images"]
    if Path(
        str(
            image_info["file_name"]
        )
    ).stem == TARGET_STEM
]


if len(target_images) != 1:
    raise RuntimeError(
        f"Expected exactly one image for "
        f"'{TARGET_STEM}', "
        f"found {len(target_images)}."
    )


target_image = target_images[0]

target_image_id = target_image[
    "id"
]


# ---------------------------------------------------------
# Get predictions belonging to this image
# ---------------------------------------------------------

target_annotations = [
    annotation
    for annotation
    in prediction_poly_coco["annotations"]
    if annotation["image_id"]
    == target_image_id
]


print(
    "Target image:",
    target_image["file_name"],
)

print(
    "Image ID:",
    target_image_id,
)

print(
    "Predicted CNT objects:",
    len(target_annotations),
)

Target image: 400-1093-w11-c1p1-befo3-10sp-15flow_left.tif
Image ID: 2
Predicted CNT objects: 168


In [5]:
# %%
OUTPUT_DIR = (
    PROJECT_ROOT
    / "notebooks"
    / "review"
    / "cpsam_pred_visualize"
    / "output"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TARGET_COCO_JSON = (
    OUTPUT_DIR
    / "target_prediction_poly.json"
)


target_coco = {
    "images": [
        target_image,
    ],
    "annotations": (
        target_annotations
    ),
    "categories": prediction_poly_coco.get(
        "categories",
        [],
    ),
}


with open(
    TARGET_COCO_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        target_coco,
        file,
        indent=2,
    )


print(
    "Temporary target COCO:",
    TARGET_COCO_JSON,
)


visualize_json_annotations_as_polygons_svg(
    json_path=TARGET_COCO_JSON,
    image_folder=IMAGE_DIR,
    output_folder=OUTPUT_DIR,
    darken_factor=0.75,
    polygon_alpha=0.9,
    contour_width=0.0,
)

Temporary target COCO: C:\Users\abd93000\PycharmProjects\cnt_project_v2\notebooks\review\cpsam_pred_visualize\output\target_prediction_poly.json
